# CLIP Distance Scene Detection

This notebook tests a scene detector based on semantic changes in CLIP image embeddings.

It does not modify the existing extraction pipeline. Outputs are written to `outputs_video_clip_distance_compare/`.

Idea:

1. Sample video frames at a fixed interval
2. Encode each sampled frame with CLIP ViT-B/32
3. Compute adjacent-frame cosine distance
4. Treat local distance peaks as cut candidates
5. Run the existing shot/text classifier and render overlay videos

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

import shotguide_batch_video_inference as base

ROOT = Path.cwd()
OUTPUT_ROOT = ROOT / 'outputs_video_clip_distance_compare'
OUTPUT_ROOT.mkdir(exist_ok=True)

TARGET_VIDEOS = [
    ROOT / 'videos' / '0065_DXzJBQKz7AK.mp4',
    ROOT / 'videos' / '0099_DS53BHjkh5o.mp4',
    ROOT / 'videos' / '0022_DXWtSqsEk71.mp4',
    ROOT / 'videos' / '0083_DUigMzGEY1J.mp4',
]

SAMPLE_INTERVAL_SEC = 0.25
DISTANCE_PERCENTILE = 88
MIN_SCENE_SEC = 0.75
MIN_DISTANCE = 0.055
NUM_FRAME_SAMPLES = 3
BATCH_SIZE = 32

for path in TARGET_VIDEOS:
    assert path.exists(), path

TARGET_VIDEOS

## 1. Load CLIP and Prediction Head

In [ ]:
clip_model, clip_preprocess, head, idx_to_shot = base.load_models()
print('device:', base.DEVICE)
print('labels:', idx_to_shot)

## 2. CLIP Distance Detector

In [ ]:
def sample_frame_indices_for_detection(video_path: Path, interval_sec=0.25):
    info = base.get_video_info(video_path)
    step = max(1, int(round(info['fps'] * interval_sec)))
    indices = list(range(0, info['frame_count'], step))
    if indices[-1] != info['frame_count'] - 1:
        indices.append(info['frame_count'] - 1)
    return indices, info

def read_frame_rgb(video_path: Path, frame_idx: int):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

def encode_detection_frames(video_path: Path, frame_indices, clip_model, clip_preprocess, batch_size=32):
    valid_indices = []
    tensors = []
    for frame_idx in tqdm(frame_indices, leave=False, desc=f'encode {video_path.stem}'):
        frame_rgb = read_frame_rgb(video_path, frame_idx)
        if frame_rgb is None:
            continue
        image = Image.fromarray(frame_rgb)
        tensors.append(clip_preprocess(image))
        valid_indices.append(frame_idx)

    features = []
    with torch.no_grad():
        for start in range(0, len(tensors), batch_size):
            batch = torch.stack(tensors[start:start + batch_size]).to(base.DEVICE)
            feat = clip_model.encode_image(batch)
            feat = feat / feat.norm(dim=-1, keepdim=True)
            features.append(feat.cpu().float().numpy())
    return np.array(valid_indices, dtype=np.int64), np.vstack(features).astype('float32')

def find_distance_peaks(distances, frame_indices, fps, percentile=88, min_scene_sec=0.75, min_distance=0.055):
    if len(distances) == 0:
        return [], float('nan')

    threshold = max(float(np.percentile(distances, percentile)), float(min_distance))
    local_peaks = []
    for i, distance in enumerate(distances):
        left = distances[i - 1] if i > 0 else -1
        right = distances[i + 1] if i + 1 < len(distances) else -1
        if distance >= threshold and distance >= left and distance >= right:
            cut_frame = int(frame_indices[i + 1])
            local_peaks.append((cut_frame, float(distance)))

    min_gap_frames = max(1, int(round(fps * min_scene_sec)))
    selected = []
    for cut_frame, distance in sorted(local_peaks, key=lambda x: x[1], reverse=True):
        if all(abs(cut_frame - old_frame) >= min_gap_frames for old_frame, _ in selected):
            selected.append((cut_frame, distance))

    selected = sorted(selected, key=lambda x: x[0])
    return selected, threshold

def build_scene_table_from_clip_peaks(video_path: Path, cut_peaks, threshold, info):
    cut_frames = [0] + [frame for frame, _ in cut_peaks]
    cut_frames = sorted(set(max(0, min(info['frame_count'] - 1, int(x))) for x in cut_frames))
    fps = info['fps']
    rows = []
    for i, start_frame in enumerate(cut_frames):
        next_start = cut_frames[i + 1] if i + 1 < len(cut_frames) else info['frame_count']
        end_frame = max(start_frame, next_start - 1)
        peak_distance = ''
        if i > 0:
            peak_distance = next((dist for frame, dist in cut_peaks if frame == start_frame), '')
        rows.append({
            'video_path': str(video_path.resolve()),
            'video_name': video_path.name,
            'video_id': video_path.stem.split('_')[0],
            'scene_index': i + 1,
            'start_frame': int(start_frame),
            'end_frame': int(end_frame),
            'start_time': start_frame / fps,
            'end_time': end_frame / fps,
            'duration_sec': (end_frame - start_frame + 1) / fps,
            'threshold': threshold,
            'peak_distance': peak_distance,
            'fps': fps,
            'detector': 'CLIP adjacent embedding distance',
        })
    return pd.DataFrame(rows)

def plot_distance_profile(video_output_dir: Path, video_path: Path, frame_indices, distances, cut_peaks, threshold, fps):
    times = np.array(frame_indices[1:]) / fps
    plt.figure(figsize=(12, 4))
    plt.plot(times, distances, linewidth=1.5)
    plt.axhline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.3f}')
    for cut_frame, distance in cut_peaks:
        plt.axvline(cut_frame / fps, color='orange', alpha=0.7)
    plt.title(f'CLIP Distance Profile: {video_path.name}')
    plt.xlabel('time (sec)')
    plt.ylabel('1 - cosine similarity')
    plt.legend()
    plt.tight_layout()
    out_path = video_output_dir / 'clip_distance_profile.png'
    plt.savefig(out_path, dpi=160)
    plt.close()
    return out_path

def detect_scenes_clip_distance(video_path: Path):
    frame_indices, info = sample_frame_indices_for_detection(video_path, SAMPLE_INTERVAL_SEC)
    valid_indices, embeddings = encode_detection_frames(video_path, frame_indices, clip_model, clip_preprocess, BATCH_SIZE)
    distances = 1.0 - np.sum(embeddings[1:] * embeddings[:-1], axis=1)
    cut_peaks, threshold = find_distance_peaks(
        distances,
        valid_indices,
        info['fps'],
        percentile=DISTANCE_PERCENTILE,
        min_scene_sec=MIN_SCENE_SEC,
        min_distance=MIN_DISTANCE,
    )
    scene_df = build_scene_table_from_clip_peaks(video_path, cut_peaks, threshold, info)
    distance_df = pd.DataFrame({
        'frame_idx': valid_indices[1:],
        'time_sec': valid_indices[1:] / info['fps'],
        'clip_distance': distances,
    })
    return scene_df, distance_df, cut_peaks, threshold, valid_indices, distances, info

## 3. Run Detection, Prediction, and Overlay

In [ ]:
summary_rows = []
all_scene_rows = []

for video_path in tqdm(TARGET_VIDEOS, desc='CLIP distance videos'):
    video_output_dir = OUTPUT_ROOT / video_path.stem
    frames_dir = video_output_dir / 'scene_frames'
    video_output_dir.mkdir(parents=True, exist_ok=True)

    scene_df, distance_df, cut_peaks, threshold, valid_indices, distances, info = detect_scenes_clip_distance(video_path)
    distance_df.to_csv(video_output_dir / 'clip_distance_profile.csv', index=False, encoding='utf-8-sig')
    plot_distance_profile(video_output_dir, video_path, valid_indices, distances, cut_peaks, threshold, info['fps'])

    scene_df = base.extract_scene_frames(video_path, scene_df, frames_dir, num_samples=NUM_FRAME_SAMPLES)
    scene_df.to_csv(video_output_dir / 'scene_metadata.csv', index=False, encoding='utf-8-sig')

    pred_df, scene_embeddings = base.predict_scenes(scene_df, clip_model, clip_preprocess, head, idx_to_shot)
    pred_df.to_csv(video_output_dir / 'scene_predictions.csv', index=False, encoding='utf-8-sig')
    np.savez_compressed(video_output_dir / 'scene_clip_embeddings.npz', embeddings=scene_embeddings)

    overlay_path = video_output_dir / f'{video_path.stem}_clip_distance_overlay.mp4'
    base.render_overlay_video(video_path, pred_df, overlay_path)

    summary_rows.append({
        'video_id': video_path.stem.split('_')[0],
        'video_name': video_path.name,
        'detector': 'CLIP adjacent embedding distance',
        'sample_interval_sec': SAMPLE_INTERVAL_SEC,
        'distance_percentile': DISTANCE_PERCENTILE,
        'min_scene_sec': MIN_SCENE_SEC,
        'threshold': threshold,
        'duration_sec': info['duration_sec'],
        'scene_count': len(pred_df),
        'avg_scene_duration_sec': float(pred_df['duration_sec'].mean()),
        'min_scene_duration_sec': float(pred_df['duration_sec'].min()),
        'max_scene_duration_sec': float(pred_df['duration_sec'].max()),
        'text_scene_count': int(pred_df['pred_has_text'].sum()),
        'shot_label_counts': json.dumps(pred_df['pred_shot_type'].value_counts().to_dict(), ensure_ascii=False),
        'overlay_path': str(overlay_path.resolve()),
    })
    all_scene_rows.append(pred_df)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / 'clip_distance_batch_summary.csv', index=False, encoding='utf-8-sig')
pd.concat(all_scene_rows, ignore_index=True).to_csv(OUTPUT_ROOT / 'clip_distance_scene_predictions.csv', index=False, encoding='utf-8-sig')

summary_df

## 4. Compare Detectors

In [ ]:
compare_rows = []
for video_path in TARGET_VIDEOS:
    old_csv = ROOT / 'outputs_video_batch_test' / video_path.stem / 'scene_predictions.csv'
    py_csv = ROOT / 'outputs_video_pyscenedetect_compare' / video_path.stem / 'scene_predictions.csv'
    clip_csv = OUTPUT_ROOT / video_path.stem / 'scene_predictions.csv'

    for detector, csv_path, overlay_suffix in [
        ('Previous adaptive frame-diff', old_csv, '_overlay.mp4'),
        ('PySceneDetect ContentDetector', py_csv, '_pyscenedetect_overlay.mp4'),
        ('CLIP adjacent embedding distance', clip_csv, '_clip_distance_overlay.mp4'),
    ]:
        if not csv_path.exists():
            continue
        pred_df = pd.read_csv(csv_path)
        overlay_dir = csv_path.parent
        compare_rows.append({
            'video_id': video_path.stem.split('_')[0],
            'video_name': video_path.name,
            'detector': detector,
            'scene_count': len(pred_df),
            'avg_scene_duration_sec': float(pred_df['duration_sec'].mean()),
            'text_scene_count': int(pred_df['pred_has_text'].sum()),
            'shot_label_counts': json.dumps(pred_df['pred_shot_type'].value_counts().to_dict(), ensure_ascii=False),
            'overlay_path': str((overlay_dir / f'{video_path.stem}{overlay_suffix}').resolve()),
        })

comparison_df = pd.DataFrame(compare_rows)
comparison_df.to_csv(OUTPUT_ROOT / 'detector_comparison_summary.csv', index=False, encoding='utf-8-sig')
comparison_df[['video_id', 'detector', 'scene_count', 'avg_scene_duration_sec', 'text_scene_count', 'shot_label_counts', 'overlay_path']]